# Food Waste Reducer - YOLO training (Colab, one click)

> **Data gate: OPEN** (ML QA PASS on #27, 2026-10-10: leak-free session/pHash split, stratification rules).
> Cell 4 still re-checks it on the data it builds here and REFUSES to train unless `split_grouping == session`,
> `ml/eval/leakage_check.py` passes (0 held-out images within 300 s of train/val; pHash near-dups <= 1% per source)
> and `ml/eval/stratification_check.py` reports no failures.
>
> **Two-step run: DRY RUN first, then the real run.** Cell 1 has a checkbox `DRY_RUN` (default **on**). A dry run does the whole
> pipeline (download, prepare, gates, cache) but trains **1 epoch on a deterministic 5% subset** (every 20th file, all three
> freshness classes kept) for both detector and classifier, exports predictions, and stops with `DRY RUN OK`. It takes about 15 min and
> catches a broken download, gate, class mapping or export before you spend 3-4.5 h. Then **untick `DRY_RUN`, keep the same `RUN_ID`**,
> and Run all: the dataset is restored from Drive (no re-download) and the real training starts. The dry run writes its checkpoints to
> `<RUN_ID>_dry/`, so they are never resumed by the real run.
>
> **Checkpoint: `last.pt`, fixed epochs** (ML-28): val has no potato and no apple/spoiled, so "best" (picked on val)
> would be blind to them. Metrics are reported on the held-out split from the last epoch.

> **If you got disconnected: reconnect, keep the same `RUN_ID`, Runtime -> Run all; it resumes from the last epoch on Drive.**
> Checkpoints (`last.pt`, written every epoch) and the prepared dataset live in Google Drive under `MyDrive/fwr_runs/<RUN_ID>/data_<DATA_VER>/` (`DATA_VER` = hash of `sources.json`, so a changed source list always rebuilds the dataset).
> Type your `RUN_ID` into the form field in cell 1 and **write it down**. Left at the default it becomes today's date, so a
> run resumed after midnight would start a fresh run instead. Finished stages are skipped, so re-running a completed notebook only re-exports.

## What the owner clicks
1. **Runtime -> Change runtime type -> T4 GPU.**
2. Left sidebar **Secrets** (key icon): add `ROBOFLOW_API_KEY` and `KAGGLE_API_TOKEN`; switch **Notebook access** on for each. (`KAGGLE_API_TOKEN` is the Kaggle access token, used as a Bearer token; optional: without it the Kaggle source is skipped. AgriFreshNET comes from the public Mendeley release and needs no secret.)
3. **Runtime -> Run all** with `DRY_RUN` ticked (default). Allow Drive access when asked: **Drive is required** (cell 1 stops with a message if it is not mounted). Wait for `DRY RUN OK`.
3b. Untick `DRY_RUN` in cell 1, keep the `RUN_ID`, **Runtime -> Run all** again for the real run.
4. When it finishes, the last cell downloads **`export.zip`** (detector.pt, freshness.pt, class_names.json, predictions.jsonl, heldout_gt.jsonl, heldout_manifest.csv, dataset_report.json). Give it to the team.

Expect about 15 min for the dry run and roughly 3-4.5 h on a free T4 for the full run (80 epochs yolo26s + 30 epochs classifier); free Colab disconnects, which is what the resume is for. Ultralytics is **AGPL-3.0**; see `docs/ML.md`. Never paste keys into cells.

In [ ]:
# 1. Setup: code, deps, Google Drive (required), run folder
import os, shutil, subprocess, sys
REPO_URL = "https://github.com/Abyss999/CodeRedOrion-CougarAIUncs.git"
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
%cd /content/repo/ml/yolo
def pip(*args):   # check=True: a failed install stops here (a `!pip` shell call would be ignored; ML-61)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:   # Colab never shows a subprocess's stderr, so print pip's own error (ML-63)
        print((e.stdout or "")[-4000:], (e.stderr or "")[-4000:])
        raise
pip("-r", "requirements.txt", "roboflow", "imagehash")
# MeatScan: unrar-cffi decompresses single files range-read from its 25 GB RAR (official UnRAR code, CRC-checked). It has no wheel
# for Python > 3.9, so it builds from source, and its setup.py fetches its setup_requires itself: that picks the newest setuptools_scm
# without its vcs_versioning dependency and crashes (ML-63). So install those build tools with pip first, pinned, and build against them.
import importlib.util
if importlib.util.find_spec("setuptools") is None:
    pip("setuptools==80.9.0")
pip("cffi==2.1.1", "setuptools_scm==8.1.0", "pytest-runner==6.0.1", "wheel==0.45.1")
pip("--no-build-isolation", "unrar-cffi==0.2.2")
pip("-r", "/content/repo/pipeline/requirements.txt")   # AgriFreshNET is ingested with `python -m pipeline` (Pillow, numpy, pyyaml)
!python -c "import ultralytics; print('ultralytics', ultralytics.__version__)"   # expect 8.4.175
import imagehash   # the pHash leakage gate needs it: fail loudly here, never skip silently
print("imagehash", imagehash.__version__)

# RUN_ID names the folder on Drive that holds checkpoints and the prepared dataset.
# KEEP IT THE SAME when you resume after a disconnect. Leave the placeholder to use today's date (then write it down).
RUN_ID = "fwr_YYYYMMDD"  #@param {type:"string"}
DRY_RUN = True  #@param {type:"boolean"}
# DRY_RUN=True: everything up to the gates, then 1 epoch on a 5% subset, then stop. Untick it (same RUN_ID) for the real run.
import datetime, re
if RUN_ID == "fwr_YYYYMMDD":
    RUN_ID = "fwr_" + datetime.date.today().strftime("%Y%m%d")
    print("RUN_ID was the placeholder, using today's date:", RUN_ID, "(write it down; type it in the form to resume after midnight)")
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", RUN_ID), "RUN_ID must start with a letter or digit and contain only letters, digits, _ and - (not empty, not . or ..)"
assert not RUN_ID.endswith("_dry"), "RUN_ID must not end in _dry: that suffix is reserved for dry-run folders and would collide with another run's <RUN_ID>_dry"

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    raise SystemExit(f"STOP: Google Drive is required (checkpoints must survive a disconnect) and could not be mounted: {e!r}\n"
                     "Re-run this cell and approve the Drive permission prompt.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("STOP: Google Drive is not mounted (/content/drive/MyDrive missing). Re-run this cell and approve the Drive prompt.")
RUNS_ROOT = "/content/drive/MyDrive/fwr_runs"
import hashlib as _h
DATA_VER = _h.sha256(open("/content/repo/ml/yolo/sources.json", "rb").read()).hexdigest()[:8]   # changes whenever a source is added or edited
CACHE_DIR = f"{RUNS_ROOT}/{RUN_ID}/data_{DATA_VER}"   # prepared-dataset cache: shared by the dry run and the real run; a new sources.json = a fresh build
print("data version (sources.json):", DATA_VER)
RUN_DIR = CACHE_DIR + ("_dry" if DRY_RUN else "")   # train.py --project: det/weights/last.pt and cls/weights/last.pt are written here every epoch.
# The dry run gets its OWN folder, so its 1-epoch checkpoints can never be resumed (or skipped as "finished") by the real run.
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(RUN_DIR, exist_ok=True)
print("DRY_RUN:", DRY_RUN, "(1 epoch on a 5% subset, then stop)" if DRY_RUN else "(real run)")
print("existing runs on Drive:", sorted(os.listdir(RUNS_ROOT)))
print("RUN_DIR:", RUN_DIR, "| det checkpoint on Drive:", os.path.exists(f"{RUN_DIR}/det/weights/last.pt"),
      "| cls checkpoint on Drive:", os.path.exists(f"{RUN_DIR}/cls/weights/last.pt"))

# Prepared dataset cache on Drive. A single tar (the dataset is ~39k small files; copying them one by one through the Drive
# mount would take far longer than the tar). The stamp file is written LAST, so a half-written copy is never trusted.
import hashlib, json, subprocess, tarfile
DATA = "/content/fwr_data"
DRIVE_TAR, DRIVE_DATA, DRIVE_SHA = f"{CACHE_DIR}/fwr_data.tar", f"{CACHE_DIR}/fwr_data", f"{CACHE_DIR}/fwr_data.sha256"
TOPDIRS = ("images", "labels", "freshness", "cls")
def _top(rel):
    p = rel.split("/")
    return p[0] if len(p) > 1 and p[0] in TOPDIRS else None
def dir_counts(root):
    """File counts per top-level dataset dir. Ultralytics' *.cache files are ignored."""
    c = dict.fromkeys(TOPDIRS, 0)
    for dp, _, fns in os.walk(root):
        for f in fns:
            t = _top(os.path.relpath(os.path.join(dp, f), root).replace(os.sep, "/"))
            if t and not f.endswith(".cache"):
                c[t] += 1
    return c
def tar_counts(tar_path):
    """Read the whole archive (fails on truncation): file counts per top-level dir (members are 'fwr_data/<dir>/...') + total members."""
    c, n = dict.fromkeys(TOPDIRS, 0), 0
    with tarfile.open(tar_path) as tf:
        for m in tf:
            n += 1
            t = _top(m.name.split("/", 1)[1]) if "/" in m.name else None
            if t and m.isfile():
                c[t] += 1
    return c, n
def make_stamp(report_path, tar_path, counts, members):
    return json.dumps({"report_sha256": hashlib.sha256(open(report_path, "rb").read()).hexdigest(),
                       "tar_bytes": os.path.getsize(tar_path), "tar_members": members, "counts": counts}, sort_keys=True)
def read_stamp():
    try:
        return json.load(open(DRIVE_SHA))
    except Exception:
        return None
_st = read_stamp()
DATA_ON_DRIVE = bool(_st and os.path.exists(DRIVE_TAR) and os.path.exists(f"{DRIVE_DATA}/dataset_report.json")
                     and _st["tar_bytes"] == os.path.getsize(DRIVE_TAR)
                     and _st["report_sha256"] == hashlib.sha256(open(f"{DRIVE_DATA}/dataset_report.json", "rb").read()).hexdigest())
print("prepared dataset on Drive, stamp OK -> skip downloads + prepare:" if DATA_ON_DRIVE else "no valid prepared dataset on Drive yet -> download + prepare:", DATA_ON_DRIVE)
if os.path.exists(DRIVE_SHA) and not DATA_ON_DRIVE:
    print("stale or invalid stamp on Drive, will rebuild")


In [ ]:
# 2. Secrets (Colab Secrets only; values are never printed)
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
for k in ("ROBOFLOW_API_KEY", "KAGGLE_API_TOKEN"):
    v = secret(k)
    if v: os.environ[k] = v
    print(k, "set" if v else "MISSING")
assert DATA_ON_DRIVE or os.environ.get("ROBOFLOW_API_KEY"), "Add ROBOFLOW_API_KEY in Colab Secrets (left sidebar) and re-run."
HAVE_KAGGLE = bool(os.environ.get("KAGGLE_API_TOKEN"))

In [ ]:
# 3. (skipped when the prepared dataset is already on Drive) Download the sources (eight active, one pending) and write a Colab sources.json derived from the repo's one
if DATA_ON_DRIVE:
    print('Prepared dataset found on Drive: skipping all downloads.')
else:
    import hashlib, json, pathlib, subprocess, sys, zipfile, urllib.request
    RAW = pathlib.Path("/content/raw"); RAW.mkdir(exist_ok=True)
    srcs = {s["name"]: s for s in json.load(open("sources.json"))["sources"]}

    # 3a. VegQual (figshare, CC BY 4.0)
    vq = RAW / "vegqual"
    if not vq.exists():
        vq.mkdir()
        urllib.request.urlretrieve("https://ndownloader.figshare.com/files/62799448", "/content/vegqual.zip")
        zipfile.ZipFile("/content/vegqual.zip").extractall(vq)
        os.remove("/content/vegqual.zip")
    srcs["vegqual"]["root"] = str(next(vq.rglob("data.yaml")).parent)

    # 3b. Roboflow (YOLO txt export). Versions are pinned in sources.json (fridge_objects v1, hien_fridge v12).
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    # Every Roboflow source that is not pending and has a pinned version (fridge_objects v1, hien_fridge v12,
    # food_spoilage_status v1 - its polygon labels become boxes in prepare_dataset). food_hvk7g stays pending (no version).
    rf_names = [n for n, s in srcs.items() if s.get("roboflow") and s["roboflow"].get("version") and not s.get("pending")]
    print("Roboflow sources:", rf_names, "| pending:", [n for n, s in srcs.items() if s.get("roboflow") and s.get("pending")])
    for name in rf_names:
        cfg = srcs[name]["roboflow"]
        dest = RAW / name
        if not dest.exists():
            rf.workspace(cfg["workspace"]).project(cfg["project"]).version(cfg["version"]).download("yolov8", location=str(dest))
        srcs[name]["root"] = str(dest); srcs[name].pop("pending", None)
        got = [l for l in open(dest / "data.yaml") if l.startswith("names")]
        print(name, "v%s" % cfg["version"], got)      # must match the class_map keys in sources.json

    # 3c. raghavrpotdar via the Kaggle REST API (Bearer token, CC0), only if the secret exists
    k = srcs["fresh_stale_fruits_veg"]
    if HAVE_KAGGLE:
        kd = RAW / "fresh_stale_fruits_veg"
        if not kd.exists():
            kd.mkdir()
            !curl -sL -H "Authorization: Bearer $KAGGLE_API_TOKEN" -o /content/kaggle.zip https://www.kaggle.com/api/v1/datasets/download/raghavrpotdar/fresh-and-stale-images-of-fruits-and-vegetables
            zipfile.ZipFile("/content/kaggle.zip").extractall(kd)
            os.remove("/content/kaggle.zip")
        k["root"] = str(next(kd.rglob("fresh_apple")).parent); k.pop("pending", None)
        print("kaggle class dirs:", sorted(p.name for p in pathlib.Path(k["root"]).iterdir() if p.is_dir()))
    else:
        k["pending"] = True
        print("KAGGLE_API_TOKEN missing: skipping fresh_stale_fruits_veg")

    # 3d. AgriFreshNET (Mendeley Data v1, CC BY 4.0, public: no secret). The file URL comes from the public API.
    #     Ingest = exclude the authors' aug_ files, cap 300 per class x freshness (round-robin over sessions/specimens),
    #     group by capture session (timestamped) / cross-stage pHash specimen (untimestamped). Same command as the local a2b run.
    AGRI_STAGED = pathlib.Path("/content/pipeline_data/staged/agrifreshnet")
    if not (AGRI_STAGED / "ingest_report.json").exists():
        # Mendeley answers HTTP 403 to urllib's default User-Agent (API and file download), so send one on BOTH requests.
        UA = {"User-Agent": "Mozilla/5.0 (compatible; fwr-colab)"}
        AGRI_SHA256 = "6607a48bf2817f9dfc8d4a1c32ce663c1d8308b4b1251e9bf65f074fcac53f11"   # Processed Data.zip, Mendeley v1 (verified 2026-10-10)
        meta = json.load(urllib.request.urlopen(urllib.request.Request("https://data.mendeley.com/public-api/datasets/42m5tb7yv9", headers=UA)))
        fl = [f for f in meta["files"] if f["filename"] == "Processed Data.zip"]
        assert len(fl) == 1, f"'Processed Data.zip' not found in the Mendeley file list: {[f['filename'] for f in meta['files']]}"
        url = fl[0]["content_details"]["download_url"]
        api_sha = fl[0]["content_details"].get("sha256_hash")
        assert api_sha in (None, AGRI_SHA256), f"Mendeley now lists a different SHA-256 ({api_sha}); the release changed. Stop and ask the team."
        print("AgriFreshNET file:", fl[0]["filename"], fl[0].get("size"), "bytes")
        h = hashlib.sha256()
        with urllib.request.urlopen(urllib.request.Request(url, headers=UA)) as r, open("/content/agrifreshnet.zip", "wb") as fh:
            for blk in iter(lambda: r.read(1 << 20), b""):
                fh.write(blk); h.update(blk)
        assert os.path.getsize("/content/agrifreshnet.zip") > 300e6, "AgriFreshNET download looks truncated (expected about 348 MB)"
        assert h.hexdigest() == AGRI_SHA256, "AgriFreshNET zip SHA-256 mismatch (corrupt download or a changed release)"   # always verified, never skipped
        subprocess.run([sys.executable, "-m", "pipeline", "--data-root", "/content/pipeline_data", "ingest-dataset",
                        "--zip", "/content/agrifreshnet.zip", "--name", "agrifreshnet", "--adapter", "agrifreshnet",
                        "--specimen-groups", "--exclude-source-augs", "--max-per-cell", "300", "--drop-raw"],
                       cwd="/content/repo", check=True)
        os.remove("/content/agrifreshnet.zip")
    ag = json.load(open(AGRI_STAGED / "ingest_report.json"))
    print("agrifreshnet staged:", ag["staged"], "(local a2b run: 4754)", "| curation before/after:", ag["curation"]["before"], "->", ag["curation"]["after"],
          "| source augs excluded:", ag["curation"]["source_augs_excluded"])
    if ag["staged"] != 4754:
        print("WARNING: staged count differs from the local reference run (4754); the gates below still decide.")
    srcs["agrifreshnet"].update(json.load(open(AGRI_STAGED / "source.json")))      # staged root + class_map (semi_fresh kept as its own label)
    srcs["agrifreshnet"].pop("pending", None)

    # 3e. MeatScan (Zenodo 16764338, CC BY 4.0): cow meat fresh/spoiled. HTTP range reads of the ORIGINALS only
    #     (the release is one 25.4 GB RAR; Zenodo is slow, so it is never downloaded whole). 450 per class spread over
    #     date x market x hour, then ingest caps 300 per cell and keeps the CSV capture time for the 300 s session split.
    MEAT_STAGED = pathlib.Path("/content/pipeline_data/staged/meatscan")
    if not (MEAT_STAGED / "ingest_report.json").exists():
        subprocess.run([sys.executable, "-m", "pipeline", "fetch-meatscan", "--out", "/content/raw/meatscan", "--per-class", "450"],
                       cwd="/content/repo", check=True)
        subprocess.run([sys.executable, "-m", "pipeline", "--data-root", "/content/pipeline_data", "ingest-dataset",
                        "--dir", "/content/raw/meatscan", "--name", "meatscan", "--adapter", "meatscan", "--max-per-cell", "300"],
                       cwd="/content/repo", check=True)
    ms = json.load(open(MEAT_STAGED / "ingest_report.json"))
    print("meatscan staged:", ms["staged"], "| adapter:", ms.get("adapter_setup"), "| excluded (augmented copies):", ms.get("excluded_by_adapter", 0))
    srcs["meatscan"].update(json.load(open(MEAT_STAGED / "source.json"))); srcs["meatscan"].pop("pending", None)

    # 3f. DaFiF fish (Mendeley vx4ptwk3pb, CC BY 4.0): labelled by days on ice (1-3 fresh, 9-11 spoiled, 4-8 dropped).
    #     300 per freshness, SHA-256 checked per file, grouped by species + session across days (same fish, one split).
    FISH_STAGED = pathlib.Path("/content/pipeline_data/staged/dafif")
    if not (FISH_STAGED / "ingest_report.json").exists():
        subprocess.run([sys.executable, "-m", "pipeline", "fetch-dafif", "--out", "/content/raw/dafif", "--max-per-cell", "300"],
                       cwd="/content/repo", check=True)
        subprocess.run([sys.executable, "-m", "pipeline", "--data-root", "/content/pipeline_data", "ingest-dataset",
                        "--dir", "/content/raw/dafif", "--name", "dafif", "--adapter", "dafif", "--max-per-cell", "300"],
                       cwd="/content/repo", check=True)
    fi = json.load(open(FISH_STAGED / "ingest_report.json"))
    print("dafif staged:", fi["staged"], "| sha mismatches:", json.load(open("/content/raw/dafif/fetch_report.json")).get("sha_mismatch") if pathlib.Path("/content/raw/dafif/fetch_report.json").exists() else "cached")
    srcs["dafif"].update(json.load(open(FISH_STAGED / "source.json"))); srcs["dafif"].pop("pending", None)

    json.dump({"sources": list(srcs.values())}, open("/content/sources.colab.json", "w"), indent=1)

In [ ]:
# 4. Build (or restore from Drive) the dataset and run the gates. Training REFUSES to start unless every gate passes.
import json
# The gates ALWAYS run, also on resume, on the restored copy of the Drive dataset. prepare_dataset.py is deterministic (seed 42,
# sha1-based group hashing, sorted traversal, seeded lighting noise) given the same pinned sources, but a restore reuses the exact bytes anyway.
if DATA_ON_DRIVE:
    shutil.rmtree(DATA, ignore_errors=True)
    subprocess.run(["tar", "-xf", DRIVE_TAR, "-C", "/content"], check=True)      # a failed extract raises; never ignored
    got = dir_counts(DATA)
    ok = (got == _st["counts"]
          and open(f"{DATA}/dataset_report.json", "rb").read() == open(f"{DRIVE_DATA}/dataset_report.json", "rb").read())
    if not ok:
        os.remove(DRIVE_SHA)       # invalidate the cache so the next Run all rebuilds it from scratch
        raise SystemExit(f"STOP: restored dataset does not match the Drive stamp (expected {_st['counts']}, got {got}). "
                         "The Drive cache was invalidated: Runtime -> Run all again to download and re-prepare.")
    print("restored from Drive:", got)
else:
    # Always a fresh build: prepare_dataset.py refuses a non-empty --out, and the gates below must never check a dataset
    # left in /content by an earlier attempt (that is how a stale copy once failed the pHash gate). A failed prepare stops here.
    shutil.rmtree(DATA, ignore_errors=True)
    subprocess.run([sys.executable, "prepare_dataset.py", "--sources", "/content/sources.colab.json", "--out", DATA], check=True)
rep = json.load(open("/content/fwr_data/dataset_report.json"))
assert rep.get("split_grouping") == "session", "Split is not session-grouped (ML-21). Stop: update the repo."
print("active classes:", rep["n_active_classes"], rep["active_classes"])
print("images:", {k: v["images"] for k, v in rep["splits"].items()})
print("dropped:", rep["dropped_images"])
print("near-duplicate drops:", rep["grouping"]["near_duplicate_drops"])
dec = rep["stratification"]["cell_decisions"]
not_meas = sorted(k for k, d in dec.items() if d["decision"] == "not_measurable_on_heldout")
print("not measurable on held-out:", {k: (dec[k]["images"], dec[k]["sessions"]) for k in not_meas})

# ML-21 gate: timestamp neighbours == 0 and pHash near-dups <= 1% per source (needs imagehash; exit code 1 on failure)
def run_check(script, json_path):
    if os.path.exists(json_path): os.remove(json_path)            # never read a stale result from an earlier or crashed run
    p = subprocess.run([sys.executable, script, DATA, "--json", json_path], capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-1500:])
    assert p.returncode == 0, f"{os.path.basename(script)} exited with {p.returncode}: gate failed or the check crashed"
    assert os.path.exists(json_path), f"{os.path.basename(script)} wrote no {json_path}"
run_check("/content/repo/ml/eval/leakage_check.py", "/content/leakage.json")
lk = json.load(open("/content/leakage.json"))["per_source"]
for s, v in lk.items():
    print(s, {x: v.get(x) for x in ("heldout_originals", "within_60s", "within_300s", "phash_near_dup", "phash_near_dup_pct")})   # .get: a train-only source (meatscan) has no pHash keys (ML-58)
    assert v["criterion"]["timestamp_300s_zero"], f"{s}: held-out image within 300 s of a train/val image"
    assert v["criterion"]["phash_le_1pct"] is True, f"{s}: pHash near-duplicates > 1% (or the check was skipped)"

# ML-23/24 gate: stratification + report consistency
run_check("/content/repo/ml/eval/stratification_check.py", "/content/strat.json")
st = json.load(open("/content/strat.json"))
assert not st["never_trained_cells"], st["never_trained_cells"]
assert not st["failures"], st["failures"]  # >=50% train, not-measurable consistency, required cells (QA rules)
assert all(v["ok"] for v in st["report_vs_manifest"].values()), "dataset_report.json disagrees with the manifest/disk"
unexpected = sorted(set(st["missing_from_heldout_required"]) - set(not_meas))
assert not unexpected, f"required cells missing from held-out: {unexpected}"
# ML-50 gate: no source photo may have files in more than one split (Roboflow rotations / flips / crop tiles of one
# parent photo). Exit code 1 on any sibling.
p = subprocess.run([sys.executable, "/content/repo/ml/eval/parent_sibling_check.py", DATA, "all"], capture_output=True, text=True)
print(p.stdout[-2000:], p.stderr[-1000:])
assert p.returncode == 0, "parent_sibling_check: a source photo has files in more than one split (or the check crashed)"
# the classifier must see all three freshness classes (A2: semi_fresh), in train and val
for sp in ("train", "val"):
    have = sorted(os.listdir(f"{DATA}/cls/{sp}"))
    assert have == ["fresh", "semi_fresh", "spoiled"], f"cls/{sp} has {have}, expected fresh, semi_fresh, spoiled"
print("ALL GATES PASSED (gate: identity pHash <= 6 and <= 1% per source; the dihedral-variant near-duplicate check is reported only, same-class with masked padding (ML-40), and is not a gate)")

# Cache the gated dataset on Drive for resume (only after the gates passed). The sha file is written last = commit marker.
if not DATA_ON_DRIVE:
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if os.path.exists(DRIVE_SHA): os.remove(DRIVE_SHA)
    part = DRIVE_TAR + ".part"
    try:
        subprocess.run(["tar", "--exclude=*.cache", "-cf", part, "-C", "/content", "fwr_data"], check=True)   # disk full / quota / I/O raises
        want, got = dir_counts(DATA), None
        got, members = tar_counts(part)                      # re-reads the archive: a truncated tar raises here
        assert got == want, f"archive file counts {got} differ from the dataset {want}"
    except BaseException:
        if os.path.exists(part): os.remove(part)
        raise
    os.replace(part, DRIVE_TAR)
    for f in ("dataset_report.json", "data.yaml", "class_names.json", "heldout_manifest.csv"):   # small, readable copy beside the tar
        shutil.copy(f"{DATA}/{f}", f"{DRIVE_DATA}/{f}")
    open(DRIVE_SHA, "w").write(make_stamp(f"{DRIVE_DATA}/dataset_report.json", DRIVE_TAR, got, members) + "\n")   # LAST = commit marker
    print("dataset cached on Drive:", DRIVE_TAR, round(os.path.getsize(DRIVE_TAR) / 1e6), "MB", got)


In [ ]:
# 5. Train (detector 80 ep yolo26s, classifier 30 ep; exports last.pt, see ML-28) and export predictions.
# --resume: finished stages are skipped, an interrupted stage continues from <RUN_DIR>/<det|cls>/weights/last.pt (saved to Drive every epoch).
# To resume after a disconnect: reconnect, keep RUN_ID, Runtime -> Run all.
SEMI_FRESH_THRESHOLD = 0.5  #@param {type:"number"}
# P(semi_fresh) a crop needs to be labelled semi_fresh in predictions.jsonl (spoiled wins first). Passed to train.py -> export_predictions.py.
# ML-57: train.py rewrites RUN_DIR/export on every run (also when --resume skips finished stages), so drop the old one first:
# cell 6 must never pass on, or download, an export left by an earlier run with the same RUN_ID. Checkpoints are kept.
shutil.rmtree(f"{RUN_DIR}/export", ignore_errors=True)
def run_train(data, epochs, cls_epochs):   # check=True: a crash in training stops here (a ! shell call would be ignored)
    subprocess.run([sys.executable, "train.py", "--data", data, "--epochs", str(epochs), "--cls-epochs", str(cls_epochs),
                    "--device", "auto", "--weights", "last", "--resume", "--semi-fresh-threshold", str(SEMI_FRESH_THRESHOLD),
                    "--project", RUN_DIR], check=True)
if DRY_RUN:
    # Deterministic 5% subset (every 20th file, all classes kept). Ultralytics' fraction=0.05 would take the FIRST 5% of the sorted
    # list (data/base.py, 8.4.175): one source / one class folder, which would not exercise the 3-class head.
    from make_subset import make_subset
    print(make_subset("/content/fwr_data", "/content/fwr_data_dry", stride=20))
    run_train("/content/fwr_data_dry", 1, 1)
else:
    run_train("/content/fwr_data", 80, 30)

In [ ]:
# 6. Quick held-out check, zip the export (it is already on Drive in RUN_DIR/export)
EXPORT = f"{RUN_DIR}/export"
if DRY_RUN:
    need = ("detector.pt", "freshness.pt", "predictions.jsonl", "heldout_gt.jsonl", "heldout_manifest.csv", "class_names.json", "dataset_report.json")
    missing = [f for f in need if not os.path.exists(f"{EXPORT}/{f}")]
    assert not missing, f"dry run export is missing {missing}"
    n_pred = sum(1 for _ in open(f"{EXPORT}/predictions.jsonl"))
    assert n_pred > 0, "dry run wrote no predictions"
    print(f"DRY RUN OK: download, prepare, gates, 3-class classifier, 1 epoch each and export ({n_pred} prediction lines) all worked.")
    print("Now untick DRY_RUN in cell 1, keep the SAME RUN_ID, and Runtime -> Run all for the real 3-4.5 h run (the dataset is restored from Drive).")
else:
    from ultralytics import YOLO
    m = YOLO(f"{EXPORT}/detector.pt")
    print(m.val(data="/content/fwr_data/data.yaml", split="test"))   # last.pt on held-out (incl. synthetic low/warm copies; only real low/warm counts for lighting, ML-17/22). The real gate run is ml/eval on predictions.jsonl
    !rm -rf /content/export_out && mkdir -p /content/export_out && cp -r "{EXPORT}" /content/export_out/export
    !cd /content/export_out && zip -r export.zip export
    from google.colab import files
    files.download("/content/export_out/export.zip")
